# SuperTrend AI 互動式測試與回測
此 Notebook 用於讀取資料、進行參數優化掃描，並產出分析報告。


In [ ]:
import sys
import os
if os.path.abspath('..') not in sys.path:
    sys.path.append(os.path.abspath('..'))

import pandas as pd
from src.supertrend_ai.optimizer import optimize_supertrend_ai
from src.supertrend_ai.reporter import generate_supertrend_ai_report
import warnings
warnings.filterwarnings('ignore')

ticker = "QQQM"
start_date = "2021-08-01"
end_date = "2026-08-01"

# 1. 取得資料 (使用原系統之 DataLoader 或 yfinance)
import yfinance as yf
df = yf.download(ticker, start=start_date, end=end_date)
# 若 yfinance 傳回 MultiIndex，需進行處理
if isinstance(df.columns, pd.MultiIndex):
    df.columns = [c[0] for c in df.columns]
    
df.index = pd.to_datetime(df.index)
df.dropna(inplace=True)


In [ ]:
# 2. 執行參數優化 (可自訂範圍)
res_df, best_params = optimize_supertrend_ai(
    df=df,
    ticker=ticker,
    start_date=start_date,
    end_date=end_date,
    lengths=[10, 14, 20],            # ATR Lengths 測試 10 與 15
    min_mults=[1.0, 1.5, 2],        # Factor 範圍起點
    max_mults=[3.0, 4.0, 5.0],        # Factor 範圍終點
    steps=[0.5],                 # Step 固定 0.5
    perf_alphas=[5, 10.0, 20],          # Performance Memory
    clusters=['Best', 'Average'],# 目標群集
    update_freqs=[1],         # KMeans 運算頻率 (1=每根K棒, 5=每5根K棒)
    long_onlys=[True]               # 探索是否只做多或多空皆做
)

print("最佳參數組合:")
for k, v in best_params.items():
    print(f"{k}: {v}")


In [ ]:
print('正在產出報告並執行詳細回測，因需逐 K 線進行 K-Means 聚類，約需 1 分鐘，請耐心等候...')
# 3. 產出 Markdown 回測報告 (含月度報酬矩陣)
from src.supertrend_ai.backtest import run_supertrend_ai_backtest

# 使用最佳參數重跑一次取得詳細紀錄
trades, equity, benchmark, df_ind, metrics = run_supertrend_ai_backtest(
    df=df,
    length=best_params['atr_length'],
    min_mult=best_params['factor_min'],
    max_mult=best_params['factor_max'],
    step=best_params['factor_step'],
    perf_alpha=best_params['perf_alpha'],
    from_cluster=best_params['from_cluster'],
    long_only=best_params.get('long_only', True),              # True=只做多, False=多空雙向交易
    show_progress=True
)

md_report = generate_supertrend_ai_report(
    ticker=ticker,
    start_date=start_date,
    end_date=end_date,
    df_ind=df_ind,
    trades=trades,
    equity=equity,
    benchmark=benchmark,
    opt_results=res_df,
    best_params=best_params,
    report_dir="data/20260802"
)

from IPython.display import display, Markdown
display(Markdown(md_report))
